In [1]:
pip install datasets transformers torch accelerate


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries
import torch
import torch.nn.functional as F
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import re
import os
import pandas as pd
import numpy as np
import time

/common/home/users/d/dzamyatin.2022/jupyterlab-venv-tf-217/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load dataset
dataset = load_dataset("gsm8k", "main")

dataset.shuffle(seed=42)

# Filter datasets
dataset = dataset.filter(
    lambda x: len(x["answer"]) > 404
)

train_dataset = dataset["train"].select(range(25))   # small first
test_dataset = dataset["test"].select(range(10))

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 25
})
Dataset({
    features: ['question', 'answer'],
    num_rows: 10
})


In [4]:
# Load model
model_name = "Qwen/Qwen2-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

# Fix pad token
# model.config.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
model.generation_config.pad_token_id = model.generation_config.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 6988.92it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [5]:
# Helper function
def extract_true_answer(answer):
    return answer.split("####")[-1].strip()

def extract_true_reasoning(answer):
    return answer.split("####")[0].strip()

def extract_predicted_answer(text):
    # take last number in output
    numbers = re.findall(r"-?\d+\.?\d*", text)
    return numbers[-1] if numbers else None

def count_tokens(input_text, output_text):
    input_tokens = len(tokenizer.encode(input_text))
    output_tokens = len(tokenizer.encode(output_text))
    return input_tokens, output_tokens

def build_prompt(question):
    return f"""You are a helpful math tutor.
Solve step by step and give the final answer.
Provide reasoning and intermediate steps and return the final numerical answer.
Output the final answer in the format on a new line: #### [final numeric answer]

Rules:
- Only provide reasoning and the final answer.
- Do not include any other text or explanations.
- Do not output code for reasoning, the reasoning must be a step by step process in natural language.
- Do not use , or . for formatting of large numbers (e.g. write 1000000 instead of 1,000,000).
- You must follow the exact format for the final answer.
- You must start directly with "Step 1:".
- Final line must be: [final numeric answer]

Response Format:
Step x: [reasoning step]
Step x + 1: [reasoning step]
...
Final Answer:[final numeric answer]

Question: {question}
Answer:"""

Reward function should reward
1. Accuracy --> answer is correct
2. Format
Step 1: [Reasoning]
Step 2: [Reasoning]
...
Final Answer: #### [Final Number]


In [6]:
# import re
def compute_format_reward(output):
    lines = [l.strip() for l in output.strip().split("\n") if l.strip()]
    reward = 0.0

    # Must start with Step 1
    if not lines[0].startswith("Step 1:"):
        reward -= 2.0

    # Extract step numbers
    step_pattern = r"Step\s*(\d+):"
    steps = [int(x) for x in re.findall(step_pattern, output)]

    if not steps:
        return -2.0

    # Penalize duplicates & wrong order
    reward -= (len(steps) - len(set(steps))) * 3.0
    expected = 1
    for s in steps:
        if s != expected:
            reward -= 3.0
            break
        expected += 1
    else:
        reward += min(len(steps) * 0.5, 2.0)

    # Final answer numeric
    final_line = lines[-1]
    if re.match(r"^Final Answer:\s*-?\d+(\.\d+)?$", final_line):
        reward += 2.0
    else:
        reward -= 2.0

    # Clip reward
    reward = max(min(reward, 3.0), -3.0)
    return reward

# def compute_format_reward(output):
#     lines = [l.strip() for l in output.strip().split("\n") if l.strip()]
    
#     if len(lines) == 0:
#         return -2.0

#     reward = 0.0

#     # =========================================================
#     # 1. Length penalty
#     # =========================================================
#     max_len = 120
#     reward -= max(0, len(output.split()) - max_len) * 0.02

#     # =========================================================
#     # 2. Must start with Step 1
#     # =========================================================
#     if not lines[0].startswith("Step 1:"):
#         reward -= 2.0

#     # =========================================================
#     # 3. Extract steps
#     # =========================================================
#     step_pattern = r"^Step\s*(\d+):"
#     step_numbers = []

#     for line in lines:
#         match = re.match(step_pattern, line)
#         if match:
#             step_numbers.append(int(match.group(1)))

#     if len(step_numbers) == 0:
#         return -2.0  # hard fail

#     # =========================================================
#     # 4. STRICT step validation (CRITICAL FIX)
#     # =========================================================
#     expected = 1
#     for step in step_numbers:
#         if step != expected:
#             # 🚨 HARD PENALTY + EARLY EXIT
#             reward -= 3.0
#             break
#         expected += 1
#     else:
#         # only reward if fully correct sequence
#         reward += min(len(step_numbers) * 0.5, 2.0)

#     # =========================================================
#     # 5. Duplicate penalty (extra safety)
#     # =========================================================
#     duplicates = len(step_numbers) - len(set(step_numbers))
#     if duplicates > 0:
#         reward -= duplicates * 1.5   # MUCH stronger

#     # =========================================================
#     # 6. Repetition penalty (token-level)
#     # =========================================================
#     words = output.split()
#     if len(words) > 0:
#         unique_ratio = len(set(words)) / len(words)
#         reward -= (1 - unique_ratio) * 2.5  # stronger

#     # =========================================================
#     # 7. Final Answer MUST be last line
#     # =========================================================
#     final_line = lines[-1]

#     if re.match(r"^Final Answer:\s*-?\d+(\.\d+)?$", final_line):
#         reward += 2.0
#     else:
#         reward -= 2.0

#     # =========================================================
#     # 8. Clip reward
#     # =========================================================
#     reward = max(min(reward, 3.0), -3.0)

#     return reward

In [ ]:
accum_steps = 5
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

baseline = 0

start_time = time.time()

for epoch in range(2):
    print(f"\nEpoch {epoch+1}")

    for i, example in enumerate(train_dataset):
        question = example["question"]

        prompt = build_prompt(question)

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=320,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.encode("Final Answer:")[0],
            bad_words_ids=[[tokenizer.encode("Step 1:")[0]]]
        )

        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        reward = compute_format_reward(response)

        # --- baseline ---
        baseline = 0.9 * baseline + 0.1 * reward
        advantage = reward - baseline

        # --- log probs ---
        full_output = outputs[0].unsqueeze(0)
        logits = model(full_output).logits
        log_probs = F.log_softmax(logits, dim=-1)

        shift_logits = log_probs[:, :-1, :]
        shift_labels = full_output[:, 1:]

        selected_log_probs = shift_logits.gather(
            2, shift_labels.unsqueeze(-1)
        ).squeeze(-1)

        gen_len = generated_tokens.shape[0]
        selected_log_probs = selected_log_probs[:, -gen_len:]

        # divide loss by accum_steps
        loss = -advantage * selected_log_probs.mean()
        loss = loss / accum_steps

        loss.backward()

        # update every 5 samples
        if (i + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

end_time = time.time()


Epoch 1

Epoch 2


In [8]:
save_path = "./qwen_format_rl"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved ✅")

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.45s/it]

Model saved ✅


In [9]:
model = AutoModelForCausalLM.from_pretrained(save_path, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(save_path, trust_remote_code=True)

model.to(device)
model.eval()

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 6652.22it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [10]:
def evaluate_format(model, dataset):
    model.eval()

    for example in dataset:
        question = example["question"]
        
        true_answer = extract_true_answer(example["answer"])
        true_reasoning = extract_true_reasoning(example["answer"])
        
        prompt = build_prompt(question)

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=320
        )

        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        model_answer = extract_predicted_answer(response)

        reward = compute_format_reward(response)

        # Accuracy
        is_correct = (float(model_answer) == float(true_answer))

        # Token usage
        input_tokens, output_tokens = count_tokens(prompt, response)
        
        results.append({
            "question":question,
            "true_reasoning": true_reasoning,
            "true_answer": true_answer,
            "model_reasoning": response,
            "model_answer": model_answer,
            "correct": is_correct,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "reward": reward
        })

In [11]:
results = []

evaluate_format(model, test_dataset)

In [12]:
df = pd.DataFrame(results)
# df.to_csv("qwen_gsm8k_results.csv", index=False)

df["true_answer"] = pd.to_numeric(df["true_answer"], errors="coerce")
df["model_answer"] = pd.to_numeric(df["model_answer"], errors="coerce")

df.to_json("rl_results.json", orient="records", indent=4)

In [13]:
total_time_min = (end_time - start_time)/60
print(total_time_min)

6.345574680964152
